# 4.3 介电常数计算器

**学习目标**
- 理解复介电常数的物理意义
- 掌握水、冰、干雪、湿雪的介电常数特性
- 学会使用混合公式计算有效介电常数
- 观察温度、频率、含水量对介电常数的影响

**参考文献**：Ryzhkov & Zrnic, Chapter 4, pp. 431-460

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown
import ipywidgets as widgets
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 理论背景

### 复介电常数

$$\varepsilon = \varepsilon' - i\varepsilon''$$

其中 $\varepsilon'$ 是实部（储能），$\varepsilon''$ 是虚部（损耗）。

### 水的介电常数 (Debye 模型)

$$\varepsilon_w = \varepsilon_{w\infty} + \frac{\varepsilon_{w0} - \varepsilon_{w\infty}}{1 + (2\pi f \tau)^2} - i\frac{(\varepsilon_{w0} - \varepsilon_{w\infty})2\pi f \tau}{1 + (2\pi f \tau)^2}$$

其中 $\varepsilon_{w0} \approx 80$，$\varepsilon_{w\infty} \approx 4$，$\tau$ 是松弛时间。

In [ ]:
def water_permittivity(T_celsius, frequency_hz):
    """
    水的复介电常数 (Debye模型)
    T_celsius: 温度 (摄氏度)
    frequency_hz: 频率 (Hz)
    """
    T_kelvin = T_celsius + 273.15
    
    # 静态介电常数
    epsilon_0 = 77.6 - 103.3 * (T_celsius / 41)
    
    # 高频极限
    epsilon_inf = 4.9
    
    # 松弛时间 (秒)
    tau = 0.000001 * np.exp(5420 * (1/T_kelvin - 1/273.15))
    
    # 角频率
    omega = 2 * np.pi * frequency_hz
    
    # Debye 方程
    epsilon_prime = epsilon_inf + (epsilon_0 - epsilon_inf) / (1 + (omega * tau)**2)
    epsilon_double = (epsilon_0 - epsilon_inf) * omega * tau / (1 + (omega * tau)**2)
    
    return epsilon_prime - 1j * epsilon_double

def ice_permittivity(T_celsius, frequency_hz):
    """
    冰的复介电常数 (简化模型)
    冰的介电损耗远小于水
    """
    epsilon_prime = 3.15  # 近似常数
    # 冰的虚部很小，这里用简化公式
    epsilon_double = 0.01  # 近似常数
    return epsilon_prime - 1j * epsilon_double

def maxwell_garnett_mixing(eps_host, eps_incl, volume_fraction):
    """
    Maxwell-Garnett 混合公式
    计算包含球形夹杂物的有效介电常数
    """
    beta = (eps_incl - eps_host) / (eps_incl + 2*eps_host)
    eps_eff = eps_host * (1 + 3*volume_fraction*beta / (1 - volume_fraction*beta))
    return eps_eff

def bruggeman_mixing(eps_1, eps_2, f_1):
    """
    Bruggeman 混合公式（双向性）
    f_1: 第一种成分的体积分数
    """
    f_2 = 1 - f_1
    # 简化计算
    term = (3*f_1 - 1)*eps_1 + (3*f_2 - 1)*eps_2
    disc = term**2 + 8*eps_1*eps_2
    eps_eff = (term + np.sqrt(disc)) / 4
    return eps_eff

def snow_permittivity(eps_ice, water_fraction, density=0.2):
    """
    雪的介电常数模型
    雪是冰和空气的混合物
    """
    eps_air = 1.0
    # 使用 Maxwell-Garnett
    eps_snow_dry = maxwell_garnett_mixing(eps_air, eps_ice, density)
    return eps_snow_dry

## 2. 介电常数计算器

In [ ]:
def plot_dielectric_calculator(material='water', T=20.0, frequency_ghz=3.0, 
                                 water_fraction=0.05, snow_density=0.2):
    """交互式介电常数计算器"""
    
    # 频率转换
    freq_hz = frequency_ghz * 1e9
    
    # 计算介电常数
    if material == 'water':
        eps = water_permittivity(T, freq_hz)
        material_name = f'纯水 (T={T}°C)'
    elif material == 'ice':
        eps = ice_permittivity(T, freq_hz)
        material_name = '冰'
    elif material == 'dry_snow':
        eps_ice = 3.15
        eps = snow_permittivity(eps_ice, 0, snow_density)
        material_name = f'干雪 (密度={snow_density})'
    elif material == 'wet_snow':
        eps_ice = 3.15
        eps_water = water_permittivity(T, freq_hz)
        # 湿雪是冰、水、空气混合物
        eps = bruggeman_mixing(eps_ice, eps_water, water_fraction)
        material_name = f'湿雪 (含水率={water_fraction*100:.1f}%)'
    elif material == 'melting_hail':
        eps_ice = 3.15
        eps_water = water_permittivity(T, freq_hz)
        eps = bruggeman_mixing(eps_ice, eps_water, water_fraction)
        material_name = f'融化冰雹 (含水率={water_fraction*100:.1f}%)'
    
    eps_prime = np.real(eps)
    eps_double = np.imag(eps)
    
    # |K|^2 计算
    K_squared = np.abs((eps - 1) / (eps + 2))**2
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 子图1: 温度对水介电常数的影响
    ax1 = axes[0, 0]
    T_range = np.linspace(-20, 40, 100)
    eps_real_range = []
    eps_imag_range = []
    for T_val in T_range:
        eps_val = water_permittivity(T_val, freq_hz)
        eps_real_range.append(np.real(eps_val))
        eps_imag_range.append(np.imag(eps_val))
    ax1.plot(T_range, eps_real_range, 'b-', linewidth=2, label="ε' (实部)")
    ax1.plot(T_range, eps_imag_range, 'r-', linewidth=2, label="ε'' (虚部)")
    ax1.axvline(x=T, color='green', linestyle='--', label=f'当前 T={T}°C')
    ax1.set_xlabel('温度 (°C)', fontsize=11)
    ax1.set_ylabel('介电常数', fontsize=11)
    ax1.set_title(f'温度对水介电常数的影响 (f={frequency_ghz} GHz)', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 子图2: 频率对水介电常数的影响
    ax2 = axes[0, 1]
    f_range = np.linspace(0.5, 30, 100)
    eps_real_f = []
    eps_imag_f = []
    for f_val in f_range:
        eps_val = water_permittivity(T, f_val*1e9)
        eps_real_f.append(np.real(eps_val))
        eps_imag_f.append(np.imag(eps_val))
    ax2.plot(f_range, eps_real_f, 'b-', linewidth=2, label="ε' (实部)")
    ax2.plot(f_range, eps_imag_f, 'r-', linewidth=2, label="ε'' (虚部)")
    ax2.axvline(x=frequency_ghz, color='green', linestyle='--', label=f'当前 f={frequency_ghz} GHz')
    ax2.set_xlabel('频率 (GHz)', fontsize=11)
    ax2.set_ylabel('介电常数', fontsize=11)
    ax2.set_title(f'频率对水介电常数的影响 (T={T}°C)', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 子图3: 当前材料的介电常数
    ax3 = axes[1, 0]
    labels = ["ε' 实部", "ε'' 虚部", '|K|²']
    values = [eps_prime, eps_double, K_squared]
    colors = ['steelblue', 'coral', 'mediumseagreen']
    bars = ax3.bar(labels, values, color=colors, alpha=0.7, edgecolor='black')
    ax3.set_ylabel('数值', fontsize=11)
    ax3.set_title(f'当前材料: {material_name}', fontsize=12)
    for bar, val in zip(bars, values):
        ax3.text(bar.get_x() + bar.get_width()/2, val + 0.5, f'{val:.3f}', ha='center')
    ax3.set_ylim(0, max(eps_prime, 5) * 1.2)
    ax3.grid(True, alpha=0.3)
    
    # 子图4: 不同材料的|K|²对比
    ax4 = axes[1, 1]
    materials_compare = ['water', 'ice', 'dry_snow']
    material_labels = ['水', '冰', '干雪']
    K_sq_values = []
    for mat in materials_compare:
        if mat == 'water':
            eps_m = water_permittivity(20, freq_hz)
        elif mat == 'ice':
            eps_m = ice_permittivity(-10, freq_hz)
        else:
            eps_m = snow_permittivity(3.15, 0, 0.2)
        K_sq_values.append(np.abs((eps_m - 1) / (eps_m + 2))**2)
    
    ax4.bar(material_labels, K_sq_values, color=['blue', 'cyan', 'lightblue'], 
            alpha=0.7, edgecolor='black')
    ax4.set_ylabel('|K|²', fontsize=11)
    ax4.set_title(f'不同材料的|K|²对比 (f={frequency_ghz} GHz)', fontsize=12)
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== {material_name} 介电特性 ===")
    print(f"频率: {frequency_ghz} GHz")
    print(f"温度: {T}°C")
    print(f"介电常数实部 ε': {eps_prime:.4f}")
    print(f"介电常数虚部 ε'': {eps_double:.4f}")
    print(f"雷达反射因子 |K|²: {K_squared:.4f}")
    print(f"复介电常数: {eps:.4f}")

In [ ]:
interact_diel = interact(plot_dielectric_calculator,
    material=widgets.Dropdown(options=[('纯水', 'water'), ('纯冰', 'ice'), 
                                      ('干雪', 'dry_snow'), ('湿雪', 'wet_snow'),
                                      ('融化冰雹', 'melting_hail')],
                             value='water', description='材料类型'),
    T=widgets.FloatSlider(min=-20, max=40, step=1, value=20, description='温度 (°C)'),
    frequency_ghz=widgets.FloatSlider(min=1, max=30, step=0.5, value=3.0, 
                                       description='频率 (GHz)'),
    water_fraction=widgets.FloatSlider(min=0, max=0.5, step=0.01, value=0.05, 
                                       description='含水体积比'),
    snow_density=widgets.FloatSlider(min=0.05, max=0.5, step=0.05, value=0.2, 
                                      description='雪密度')
)

## 3. 练习题

1. **水vs冰**：在X波段（9.4GHz）和20°C条件下，水和冰的|K|²比值约为多少？

2. **温度效应**：水的介电常数实部如何随温度变化？这对雷达探测有何影响？

3. **湿雪影响**：湿雪的|K|²比干雪大还是小？为什么？

4. **频率选择**：为什么在暴雨中S波段比X波段衰减更小？

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 4, Springer.
- Ulaby, F.T., et al., 1986: Microwave Remote Sensing, Artech House.